# Laboratório 06 — Agrupamento e PCA

**Fundamentos de Inteligência Artificial** · 2026.2 · Prof. Leonardo Garcia Marques

---

Até o lab05 todo conjunto de dados tinha **rótulo**: a resposta certa estava numa
coluna, e dava para medir o acerto. Hoje não. Vamos agrupar **178 vinhos** a partir
de **13 medidas químicas**, sem dizer ao algoritmo de que tipo cada vinho é.

Os dados vêm **dentro do Scikit-Learn** (`load_wine`): nada é baixado. O notebook
roda no laboratório, em casa e no Google Colab.

O conjunto tem, sim, uma coluna com o tipo de cada vinho (são 3 tipos). Nós a
**guardamos de lado** na Parte 1 e só olhamos para ela na **Parte 9**, no fim. É a
única vez no semestre em que um agrupamento pode ser conferido: na vida real, essa
coluna não existe.

| Parte | O que |
|---|---|
| 1 | carregar e esconder o rótulo |
| 2 | olhar as escalas |
| 3 | k-means nos dados crus |
| 4 | padronizar e repetir |
| 5 | escolher $k$: inércia e silhueta |
| 6 | o k-means em dados sem grupo nenhum |
| 7 | o sorteio inicial |
| 8 | PCA: variância explicada e o desenho em 2D |
| 9 | só agora: conferir com os tipos |

As lacunas estão marcadas com `# TODO`. Cada uma tem uma dica e um valor provisório,
para o notebook rodar inteiro mesmo antes de você preencher. Os números
**(esperado: ...)** são os que você deve ver depois de preencher: a semente é fixa,
então todos chegam aos mesmos números.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

SEMENTE = 20261007

## Parte 1 — Carregar, e esconder o rótulo

`X` fica com as 13 medidas. O tipo de cada vinho vai para `tipos_escondidos`, e
**nenhuma célula até a Parte 9 usa essa variável**.

In [ ]:
dados = load_wine(as_frame=True)
X = dados.data
tipos_escondidos = dados.target      # só na Parte 9

print("vinhos:", X.shape[0], "  medidas:", X.shape[1])

## Parte 2 — Olhar as escalas

O k-means mede **distância**. Antes de rodar, vale ver em que escala cada coluna
está.

In [ ]:
X.describe().T[["min", "max", "std"]].round(2)

In [ ]:
# TODO 1: qual coluna tem o MAIOR desvio-padrão?
# Dica: X.std() devolve o desvio de cada coluna; .idxmax() devolve o nome da maior.
maior = "?"   # valor provisório

print("coluna de maior desvio:", maior, "  (esperado: proline)")

**Pergunte-se:** o desvio da prolina é mais de mil vezes o das cinzas (`ash`).
Quando o k-means somar as diferenças ao quadrado das 13 colunas, quem vai decidir a
conta?

## Parte 3 — k-means nos dados crus

`n_init=10`: o Scikit-Learn roda o algoritmo dez vezes, com sorteios iniciais
diferentes, e fica com a menor inércia (veremos por que na Parte 7).

In [ ]:
km_cru = KMeans(n_clusters=3, n_init=10, random_state=SEMENTE).fit(X)
print(f"inércia: {km_cru.inertia_:.2f}   (esperado: 2370689.69)")

In [ ]:
# TODO 2: calcule a silhueta média do agrupamento.
# Dica: silhouette_score(dados, rotulos_dos_grupos). Os grupos estão em km_cru.labels_.
sil_cru = float("nan")   # valor provisório

print(f"silhueta: {sil_cru:.4f}   (esperado: 0.5711)")

Uma silhueta de 0,57 parece boa. Guarde esse número: na Parte 9 veremos que
esses grupos são **ruins**.

## Parte 4 — Padronizar e repetir

O `StandardScaler` deixa cada coluna com média 0 e desvio 1. Nenhuma coluna manda
mais que as outras só por estar em números maiores.

In [ ]:
# TODO 3: padronize X.
# Dica: StandardScaler().fit_transform(X)
Z = X.values   # valor provisório: ainda NÃO padronizado

print("médias  :", np.round(Z.mean(axis=0)[:4], 2), "...  (esperado: 0.0 em todas)")
print("desvios :", np.round(Z.std(axis=0)[:4], 2), "...  (esperado: 1.0 em todas)")

In [ ]:
km_pad = KMeans(n_clusters=3, n_init=10, random_state=SEMENTE).fit(Z)
sil_pad = silhouette_score(Z, km_pad.labels_)
print(f"inércia : {km_pad.inertia_:.2f}   (esperado: 1277.93)")
print(f"silhueta: {sil_pad:.4f}   (esperado: 0.2849)")

A inércia caiu de milhões para pouco mais de mil, e a silhueta **caiu** para
menos da metade. Nenhum dos dois números pode ser comparado com os da Parte 3: as
distâncias agora estão em outra escala. **Inércia e silhueta só comparam
agrupamentos dos mesmos dados.**

## Parte 5 — Quantos grupos? Inércia e silhueta para $k$ de 2 a 8

In [ ]:
ks = range(2, 9)
inercias, silhuetas = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=SEMENTE).fit(Z)
    # TODO 4: guarde a inércia e a silhueta deste k nas duas listas.
    # Dica: km.inertia_ e silhouette_score(Z, km.labels_)
    inercias.append(float("nan"))    # valor provisório
    silhuetas.append(float("nan"))   # valor provisório

print(pd.DataFrame({"k": list(ks), "inércia": inercias, "silhueta": silhuetas}).round(4).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(list(ks), inercias, "o-"); ax[0].set_title("inércia"); ax[0].set_xlabel("k")
ax[1].plot(list(ks), silhuetas, "o-", color="tab:red"); ax[1].set_title("silhueta média"); ax[1].set_xlabel("k")
plt.show()

melhor = list(ks)[int(np.nanargmax(silhuetas))] if not np.all(np.isnan(silhuetas)) else None
print("k de maior silhueta:", melhor, "  (esperado: 3)")

**Responda:** onde está o cotovelo da inércia? A silhueta concorda? Por que não
se escolhe o $k$ de **menor** inércia?

## Parte 6 — E se não houver grupo nenhum?

300 pontos sorteados ao acaso num quadrado. Não há estrutura nenhuma.

In [ ]:
rng = np.random.default_rng(SEMENTE)
U = rng.uniform(0, 10, size=(300, 2))
km_u = KMeans(n_clusters=3, n_init=10, random_state=SEMENTE).fit(U)
sil_u = silhouette_score(U, km_u.labels_)

plt.scatter(U[:, 0], U[:, 1], c=km_u.labels_, s=14, cmap="tab10")
plt.gca().set_aspect("equal"); plt.title("k = 3 em pontos sorteados"); plt.show()
print(f"silhueta do ruído: {sil_u:.4f}   (esperado: 0.4051)")

O k-means **sempre** devolve $k$ grupos. E a silhueta do ruído (0,41) é maior
que a do vinho padronizado (0,28). A silhueta serve para comparar valores de $k$
nos **mesmos dados**; ela não prova que existem grupos.

## Parte 7 — O sorteio inicial

Com `init="random"` e `n_init=1`, o k-means roda **uma vez**, a partir de um sorteio.
Dez sementes, $k = 5$:

In [ ]:
ins = []
for s in range(10):
    km = KMeans(n_clusters=5, init="random", n_init=1, random_state=s).fit(Z)
    ins.append(km.inertia_)
print("inércias:", [round(v, 2) for v in ins])
print(f"menor: {min(ins):.2f}   (esperado: 1110.67)")
print(f"maior: {max(ins):.2f}   (esperado: 1188.84)")

Todas as dez execuções **pararam**: em cada uma, nenhum ponto trocava mais de
grupo. Mas pararam em lugares diferentes. É um **mínimo local**, e é por isso que se
usa `n_init` maior que 1.

## Parte 8 — PCA

O PCA acha as direções em que os dados mais variam (Géron 2019, cap. 8, p. 215).
A fração da variância que cada componente guarda está em
`explained_variance_ratio_` (p. 218).

In [ ]:
pca_cru = PCA().fit(X)
pca_pad = PCA().fit(Z)
print(f"1º componente, dados crus      : {pca_cru.explained_variance_ratio_[0]:.4f}   (esperado: 0.9981)")
print(f"1º componente, dados padronizados: {pca_pad.explained_variance_ratio_[0]:.4f}   (esperado: 0.3620)")

Nos dados crus, um componente só guarda quase toda a variância. Não é que os
vinhos sejam simples: é a prolina, de novo.

In [ ]:
acumulada = np.cumsum(pca_pad.explained_variance_ratio_)
print("acumulada:", np.round(acumulada, 4))

# TODO 5: quantos componentes são precisos para guardar pelo menos 95% da variância?
# Dica: o livro faz assim (p. 219): np.argmax(acumulada >= 0.95) + 1
d95 = 0   # valor provisório

print(f"2 componentes guardam: {acumulada[1]:.4f}   (esperado: 0.5541)")
print("componentes para 95%:", d95, "  (esperado: 10)")

In [ ]:
# TODO 6: projete Z nos 2 primeiros componentes e desenhe, colorindo pelo grupo do k-means.
# Dica: PCA(n_components=2).fit_transform(Z)
Z2 = Z[:, :2]   # valor provisório: só as duas primeiras colunas

plt.scatter(Z2[:, 0], Z2[:, 1], c=km_pad.labels_, s=18, cmap="tab10")
plt.xlabel("1º componente"); plt.ylabel("2º componente"); plt.title("vinhos em 2D, cor = grupo"); plt.show()

Este gráfico mostra 55% da variação dos vinhos. É um retrato honesto e
incompleto: dois pontos próximos no desenho podem estar longe nas 13 dimensões.

## Parte 9 — Só agora: conferir com os tipos

Esta é a coluna que guardamos na Parte 1. Os números dos grupos (0, 1, 2) são
arbitrários: o grupo 0 do k-means não tem nada a ver com o tipo 0. Por isso, para
cada grupo, contamos quantos vinhos são do tipo mais comum nele.

In [ ]:
def coincidem(rotulos):
    tabela = pd.crosstab(tipos_escondidos, rotulos)
    return int(tabela.max(axis=0).sum()), tabela

n_pad, tab_pad = coincidem(km_pad.labels_)
n_cru, tab_cru = coincidem(km_cru.labels_)
print("padronizado:"); print(tab_pad)
print(f"vinhos que coincidem (de 178): {n_pad}   (esperado: 172)")
print()
print("cru:"); print(tab_cru)
print(f"vinhos que coincidem (de 178): {n_cru}   (esperado: 125)")

Os grupos com silhueta 0,57 (dados crus) coincidem com os tipos em 70% dos
vinhos; os de silhueta 0,28 (padronizados), em 97%. **Padronize antes.**

## Para entregar a si mesmo (não vale nota)

1. Por que o k-means precisa que você diga o $k$?
2. Por que a inércia não serve, sozinha, para escolher $k$?
3. O que a Parte 6 mostra sobre a silhueta?
4. Por que as dez execuções da Parte 7 deram inércias diferentes, se todas
   "terminaram"?
5. Por que o 1º componente dos dados crus guarda quase toda a variância?
6. Na Parte 9 usamos os tipos. Por que, num problema real de agrupamento, essa
   conferência não existe?